
# PROCESSAMENTO DE DADOS CVM — DFP / ITR

TCC: Predição de Indicadores Financeiros com Machine Learning

Etapa atual (CRISP-DM): Compreensão dos Dados
Objetivo  : 
- Ingestão dos ZIPs da CVM + EDA completa para identificar contas disponíveis, 
cobertura temporal, qualidade dos dados e KPIs financeiros relevantes para a etapa de modelagem.

Dependências: pandas, numpy, matplotlib, seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import zipfile
import os
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# 1. Configurações

In [ ]:
PASTAS        = ['./TCC_dados/DFP', './TCC_dados/ITR']
ARQUIVO_SAIDA = 'cvm_dados_processados.csv'

TIPOS_DEMONSTRATIVOS = [
    'BPA_con',    # Balanço Patrimonial Ativo
    'BPP_con',    # Balanço Patrimonial Passivo
    'DRE_con',    # Demonstração do Resultado
    'DFC_MD_con', # Fluxo de Caixa — Método Direto
    'DFC_MI_con', # Fluxo de Caixa — Método Indireto (Energia, Commodities)
    'DVA_con',    # Demonstração do Valor Adicionado
    'DMPL_con',   # Mutações do Patrimônio Líquido
    'DRA_con',    # Resultado Abrangente
]

EMPRESAS_POR_SETOR = {
    'Petróleo': [
        '33.000.167/0001-01',  # Petrobras
        '07.354.482/0001-42',  # Prio
        '33.256.439/0001-08',  # Ultrapar
        '33.453.598/0001-23',  # Raízen
        '34.274.233/0001-02',  # Vibra Energia
    ],
    'Energia': [
        '02.474.103/0001-40',  # Engie Brasil
        '03.220.438/0001-73',  # Equatorial Energia
        '07.859.971/0001-14',  # Taesa
        '02.429.144/0001-93',  # CPFL Energia
        '02.998.611/0001-04',  # ISA CTEEP
    ],
    'Varejo': [
        '92.754.738/0001-71',  # Lojas Renner
        '47.960.950/0001-21',  # Magazine Luiza
        '61.079.117/0001-05',  # Alpargatas
        '16.590.234/0001-48',  # Arezzo
        '24.990.777/0001-09',  # Grupo Mateus
    ],
    'Commodities': [
        '33.592.510/0001-54',  # Vale
        '33.359.392/0001-75',  # Gerdau
        '16.404.287/0001-55',  # Suzano
        '89.637.490/0001-45',  # Klabin
        '51.503.388/0001-43',  # São Martinho
    ],
    'Tecnologia': [
        '84.429.695/0001-11',  # WEG
        '53.113.791/0001-22',  # Totvs
        '02.351.877/0001-52',  # Locaweb
        '07.689.002/0001-89',  # Embraer
        '16.670.085/0001-55',  # Localiza
    ],
}

CNPJS_FILTRO    = [c for setor in EMPRESAS_POR_SETOR.values() for c in setor]
CNPJ_PARA_SETOR = {c: s for s, cs in EMPRESAS_POR_SETOR.items() for c in cs}

#----------------------------------------------------------------------------
# Mapa de contas CVM → nomes legíveis
# Usado na EDA para rotular colunas com nomes de relatórios financeiros reais
# Fonte: estrutura padrão de DREs e BPs consolidados da CVM (COSIF/ITG 1000)
# -----------------------------------------------------------------------------
MAPA_CONTAS = {
    # DRE
    'DRE_3.01': 'Receita Líquida',
    'DRE_3.02': 'Custo dos Produtos/Serviços',
    'DRE_3.03': 'Lucro Bruto',
    'DRE_3.04': 'Despesas Operacionais',
    'DRE_3.05': 'EBIT',
    'DRE_3.06': 'Resultado Financeiro Líquido',
    'DRE_3.07': 'Resultado antes do IR/CSLL',
    'DRE_3.08': 'IR e CSLL',
    'DRE_3.09': 'Resultado de Operações Descont.',
    'DRE_3.10': 'Lucro/Prejuízo Consolidado',
    'DRE_3.11': 'Lucro Líquido do Período',
    # BPA
    'BPA_1':       'Ativo Total',
    'BPA_1.01':    'Ativo Circulante',
    'BPA_1.01.01': 'Caixa e Equiv. de Caixa',
    'BPA_1.01.02': 'Aplicações Financeiras CP',
    'BPA_1.01.03': 'Contas a Receber',
    'BPA_1.01.04': 'Estoques',
    'BPA_1.02':    'Ativo Não Circulante',
    'BPA_1.02.01': 'Aplicações Financeiras LP',
    'BPA_1.02.03': 'Imobilizado',
    'BPA_1.02.04': 'Intangível',
    # BPP
    'BPP_2':       'Passivo Total + PL',
    'BPP_2.01':    'Passivo Circulante',
    'BPP_2.01.04': 'Empréstimos CP',
    'BPP_2.02':    'Passivo Não Circulante',
    'BPP_2.02.01': 'Empréstimos LP',
    'BPP_2.03':    'Patrimônio Líquido',
    'BPP_2.03.01': 'Capital Social',
    'BPP_2.03.05': 'Lucros/Prejuízos Acumulados',
    # DFC
    'DFC_MD_6.01': 'FC Operacional (MD)',
    'DFC_MD_6.02': 'FC de Investimento (MD)',
    'DFC_MD_6.03': 'FC de Financiamento (MD)',
    'DFC_MI_6.01': 'FC Operacional (MI)',
    'DFC_MI_6.02': 'FC de Investimento (MI)',
    'DFC_MI_6.03': 'FC de Financiamento (MI)',
    # DVA
    'DVA_7.08':    'Valor Adicionado Total',
    # D&A e EBITDA — calculados, não são contas CVM diretas
    'DA_TOTAL':      'Deprec. & Amortização (DFC_MI)',
    'EBITDA':        'EBITDA',
    'MARGEM_EBITDA_%': 'Margem EBITDA (%)',
    'MARGEM_BRUTA_%':  'Margem Bruta (%)',
    'MARGEM_EBIT_%':   'Margem EBIT (%)',
    'MARGEM_LIQUIDA_%':'Margem Líquida (%)',
    'ROE_%':           'ROE (%)',
    'ROA_%':           'ROA (%)',
    'LIQUIDEZ_CORRENTE':'Liquidez Corrente',
    'LIQUIDEZ_IMEDIATA':'Liquidez Imediata',
    'ENDIVIDAMENTO_%': 'Endividamento (%)',
    'ALAVANCAGEM_DE':  'Alavancagem D/E',
    'DIVIDA_LIQUIDA':  'Dívida Líquida',
    'COBERTURA_JUROS': 'Cobertura de Juros',
    'GIRO_ATIVO':      'Giro do Ativo',
    'FCO_SOBRE_RECEITA_%': 'FCO/Receita (%)',
    'FCO_SOBRE_LL_%':  'FCO/Lucro Líq. (%)',
}

# 2. Funções de ingestão